In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
from alphagenome_pytorch.plotting.splicing import (
    plot_splice_site_dynamics,
    plot_splice_site_predictions,
    plot_usage_density,
    plot_auprc_by_category_all_species,
    plot_auprc_by_category_all_species_summary,
    plot_pearson_r_by_category_all_species,
    SPLICE_CLASS_NAMES,
    BACKGROUND_CLASS,
    CLASS_LABELS,
    CLASS_COLORS,
    TISSUE_COLORS,
    TISSUE_ORDER,
    SPECIES_ORDER,
    SPECIES,
    SPECIES_SCI,
    CHR_SIZES,
)
from alphagenome_pytorch.evaluation.splicing import (
    build_interval_index,
    ovl_feature_vectorized,
    fast_auprc,
    load_species_data,
    get_categories_by_auprc,
    load_pearson_r_data,
    get_categories_by_pearson_r,
)

In [ ]:
work_dir = "/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/"
ann_data_dir = "/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/data/"
model = "ft_human_mouse_rat_rabbit_opossum"
model_dir = os.path.join(work_dir, model)

subsets = ["intersect_protein_coding"] # "intersect_usage", "intersect_protein_coding", "gtf_protein_coding",
subset = subsets[0]
pred_dir = f"preds_{subset}"
conf_fn = f"data_config_{subset}.json"
data_config_path = os.path.join(ann_data_dir, conf_fn)
data_config = json.load(open(data_config_path, "r"))

Annotate splice sites with the transcript biotype they overlap.

## Genomic features overlap


Load annotations for the test data

In [ ]:
sp = "opossum"

gtf_fn = data_config.get(sp).get("gene_annotation")
gtf_fn = os.path.expanduser(gtf_fn)
if os.path.exists(gtf_fn):
    gtf_df = pd.read_parquet(gtf_fn)
    print(f"Loaded GTF dataframe from {gtf_fn} with {len(gtf_df)} rows")
else:
    print(f"GTF file not found: {gtf_fn}")
    gtf_df = None

display(gtf_df)

ann_fn = data_config.get(sp).get("annotation_parquet")
ann_fn = os.path.expanduser(ann_fn)
if os.path.exists(ann_fn):
    ann_df = pd.read_parquet(ann_fn)
    print(f"Loaded annotation dataframe from {ann_fn} with {len(ann_df)} rows")
else:
    print(f"Annotation parquet file not found: {ann_fn}")
    ann_df = None

display(ann_df)

In [ ]:
gtf_df.columns

This function overlapps each splice site with annnotated features from the GTF file.

This takes ~6 mins for human, 5 min for mouse.

In [ ]:
%%time

# Rebuild index for the new species
gtf_index = build_interval_index(gtf_df)

# Make a working copy of ann_df to avoid modifying the original
ann_df_work = ann_df.copy().reset_index(drop=True)

# Apply the vectorized overlap function to annotate features for all sites in ann_df_work
ann_df_work[['Overlapping_Feature', 'gene_biotype', 'gene', 'transcript', 'exon']] = ovl_feature_vectorized(ann_df_work, gtf_df, gtf_index)

# Save the annotated dataframe to a new parquet file
#ann_out_fn = ann_fn.replace(".parquet", "_with_gtf_annotation.parquet")
#ann_df_work.to_parquet(ann_out_fn, index=False)
#print(f"Saved annotated dataframe to {ann_out_fn}")

ann_df_work

## Genomic features overlap for splice site classification predictions

### Calculate per species

In [ ]:
sp = "opossum"

preds = os.path.join(work_dir, model, "preds_intersect_protein_coding", sp, f"predictions_{sp}.parquet")
preds_df = pd.read_parquet(preds)
preds_df = preds_df.rename(columns={"chrom": "Chromosome", "position": "Position"})
display(preds_df)

annot = os.path.join(ann_data_dir, SPECIES_SCI[sp], "splice_sites_intersect_protein_coding_with_gtf_annotation.parquet")
annot_df = pd.read_parquet(annot)
display(annot_df)

# Add annotations to preds_df by merging on Chromosome and Position
preds_annot_df = preds_df.merge(
    annot_df,
    on=['Chromosome', 'Position'],
    how='left'
)

# Split by ; and order alphabetically to group similar categories together, after removing gene and transcript (they appear everywhere)
def _sort_overlap_feature(x):
    if not isinstance(x, str) or pd.isna(x):
        return np.nan
    parts = [f for f in x.split(";") if f not in {"gene", "transcript"}]
    return ";".join(sorted(parts))

preds_annot_df["Overlapping_Feature_Sorted"] = preds_annot_df["Overlapping_Feature"].apply(_sort_overlap_feature)
display(preds_annot_df[preds_annot_df['class_label'] != 4])

# Count number of sites in each category of Overlapping_Feature_Sorted
feature_counts = preds_annot_df[preds_annot_df['class_label'] != 4]['Overlapping_Feature_Sorted'].value_counts(dropna=False)
print(feature_counts)

Calculate AUPRC per class for each category. This takes ~6 mins for human.

In [ ]:
%%time

# Sample background once, reuse across all categories
bg_df = preds_annot_df[preds_annot_df['class_label'] == 4]
MAX_BG = len(bg_df)
if len(bg_df) > MAX_BG:
    bg_df = bg_df.sample(MAX_BG, random_state=1950)

# Categories we are interested in:
cats = [
    "CDS;exon",
    "CDS;exon;three_prime_utr",
    "CDS;exon;five_prime_utr",
    "CDS;exon;five_prime_utr;three_prime_utr",
    "exon",
    "exon;five_prime_utr",
    "exon;three_prime_utr",
    "exon;five_prime_utr;three_prime_utr"
]

# Precompute bg arrays per class
bg_probs = {c: bg_df[f'prob_{c}'].values for c in range(4)}
bg_labels = {c: np.zeros(len(bg_df), dtype=np.int8) for c in range(4)}

auprc_results = []
for cat in cats:
    fg_df = preds_annot_df[preds_annot_df['Overlapping_Feature_Sorted'] == cat]

    for class_label in range(4):
        fg_true = (fg_df['class_label'] == class_label).values.astype(np.int8)

        if fg_true.sum() == 0:
            continue

        fg_probs = fg_df[f'prob_{class_label}'].values

        # Concatenate fg + bg as numpy arrays (no DataFrame overhead)
        y_true   = np.concatenate([fg_true,               bg_labels[class_label]])
        y_scores = np.concatenate([fg_probs,              bg_probs[class_label]])

        auprc = fast_auprc(y_true, y_scores)

        auprc_results.append({
            'category':    cat,
            'class_label': class_label,
            'auprc':       auprc,
            'n_samples':   len(y_true),
            'n_positives': int(fg_true.sum()),
        })
        print(f"{cat} | class {class_label} | AUPRC={auprc:.4f} | n_pos={fg_true.sum()}")

auprc_df = pd.DataFrame(auprc_results)

auprc_df['baseline'] = auprc_df['n_positives'] / auprc_df['n_samples']
auprc_df['auprc_norm'] = (auprc_df['auprc'] - auprc_df['baseline']) / (1 - auprc_df['baseline'])

# Save
auprc_out_fn = preds.replace(".parquet", "_auprc_by_feature_category.csv")
auprc_df.to_csv(auprc_out_fn, index=False)
print(f"Saved AUPRC results to {auprc_out_fn}")

display(auprc_df)

### Plot for all species

In [ ]:
BASE_DIR = os.path.join(model_dir, pred_dir)
species_dfs = load_species_data(SPECIES, BASE_DIR, blacklist=['exon', 'CDS;exon;five_prime_utr;three_prime_utr', 'exon;five_prime_utr;three_prime_utr'])
sorted_cats = get_categories_by_auprc(species_dfs, min_positives=0)
print(f"Shared categories ({len(sorted_cats)}): {sorted_cats}")


In [ ]:
colors_cats = {
    "CDS;exon": "#a8a8a8",
    "CDS;exon;three_prime_utr": "#14b4f3",
    "CDS;exon;five_prime_utr": "#d62728",
    "CDS;exon;five_prime_utr;three_prime_utr": "#e377c2",
    "exon": "#cdcdcd",
    "exon;five_prime_utr": "#ff9f9f",
    "exon;three_prime_utr": "#74d8ff",
    "exon;five_prime_utr;three_prime_utr": "#ffc5ed"
}

plot_auprc_by_category_all_species(species_dfs, sorted_cats, min_positives=0)
plot_auprc_by_category_all_species_summary(species_dfs, sorted_cats, min_positives=0, colors_cats=colors_cats)

## Genomic features overlap for splice usage predictions

### Calculate per species

In [ ]:
sp = "human"

for sp in ["human", "mouse", "rat", "rabbit", "opossum"]:
    preds = os.path.join(work_dir, model, "preds_intersect_protein_coding", sp, f"usage_{sp}.parquet")
    preds_df = pd.read_parquet(preds)
    display(preds_df)

    annot = os.path.join(ann_data_dir, SPECIES_SCI[sp], "splice_sites_intersect_protein_coding_with_gtf_annotation.parquet")
    annot_df = pd.read_parquet(annot)
    display(annot_df)

    # Add annotations to preds_df by merging on Chromosome and Position
    preds_annot_df = preds_df.merge(
        annot_df,
        on=['Chromosome', 'Position'],
        how='left'
    )

    # Split by ; and order alphabetically to group similar categories together, after removing gene and transcript (they appear everywhere)
    def _sort_overlap_feature(x):
        if not isinstance(x, str) or pd.isna(x):
            return np.nan
        parts = [f for f in x.split(";") if f not in {"gene", "transcript"}]
        return ";".join(sorted(parts))

    preds_annot_df["Overlapping_Feature_Sorted"] = preds_annot_df["Overlapping_Feature"].apply(_sort_overlap_feature)

    # Count number of sites in each category of Overlapping_Feature_Sorted
    feature_counts = preds_annot_df['Overlapping_Feature_Sorted'].value_counts(dropna=False)
    print(feature_counts)

    # Calculate Pearson R between true_usage and pred_usage for each categoryof Overlapping_Feature_Sorted
    corr_results = []
    for cat in preds_annot_df['Overlapping_Feature_Sorted'].unique():
        sub = preds_annot_df[preds_annot_df['Overlapping_Feature_Sorted'] == cat]
        if len(sub) < 10:
            continue
        corr = sub[['SSE_true', 'SSE_pred']].corr().iloc[0, 1]
        nsts = sub[['Chromosome', 'Position']].drop_duplicates().shape[0]
        corr_results.append({'category': cat, 'pearson_r': corr, 'n_sites': nsts})
        print(f"{cat}: Pearson R={corr:.4f} (n={nsts})")

    corr_df = pd.DataFrame(corr_results).sort_values('pearson_r', ascending=False)

    # Save
    corr_out_fn = preds.replace(".parquet", "_pearson_r_by_feature_category.csv")
    corr_df = pd.DataFrame(corr_results).sort_values('pearson_r', ascending=False)
    corr_df.to_csv(corr_out_fn, index=False)
    print(f"Saved correlation results to {corr_out_fn}")

    display(corr_df)

### Plot for all species

In [ ]:
BASE_DIR = os.path.join(model_dir, pred_dir)
species_dfs = load_pearson_r_data(SPECIES, BASE_DIR, blacklist=['exon', 'CDS;exon;five_prime_utr;three_prime_utr', 'exon;five_prime_utr;three_prime_utr'])
sorted_cats = get_categories_by_pearson_r(species_dfs, min_positives=0)
print(f"Shared categories ({len(sorted_cats)}): {sorted_cats}")

In [ ]:
colors_cats = {
    "CDS;exon": "#a8a8a8",
    "CDS;exon;three_prime_utr": "#14b4f3",
    "CDS;exon;five_prime_utr": "#d62728",
    #"CDS;exon;five_prime_utr;three_prime_utr": "#e377c2",
    #"exon": "#cdcdcd",
    "exon;five_prime_utr": "#ff9f9f",
    "exon;three_prime_utr": "#74d8ff",
    #"exon;five_prime_utr;three_prime_utr": "#ffc5ed"
}
cats_to_plot = colors_cats.keys()  # or sorted_cats if you want all

plot_pearson_r_by_category_all_species(species_dfs, cats_to_plot, min_positives=0, colors_cats=colors_cats)